In [1]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from sklearn.preprocessing import MinMaxScaler
import matplotlib.pyplot as plt
import glob
import copy
import os

In [ ]:
class Subnetwork(nn.Module):
    def __init__(self, input_dim, output_dim = 1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 256), nn.ReLU(),
            nn.Linear(256, 256), nn.ReLU(),
            nn.Linear(256, 1)
        )

    def forward(self, x):
        return self.net(x)

    def train(model, dataloader, epochs = 2000, patience = 80, min_delta = 1e-6):
        criterion = nn.L1Loss()
        optimizer = optim.Adam(model.parameters(), lr=5e-4)
        scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma = 0.95)

        best_loss = float('inf')
        patience_counter = 0
        best_weights = copy.deepcopy(model.state_dict())

        model.train()
        print(f"Trainig has been started...\n Epochs: {epochs} \n Patience: {patience}")

        for epoch in range(epochs):
            epoch_loss = 0

            for u_seq, y_true_seq in dataloader:
                x_init = torch.zeros(u_seq.size(0), model.num_states)
                optimizer.zero_grad()
                y_pred_seq = model(u_seq, x_init)
                loss = criterion(y_pred_seq.view_as(y_true_seq), y_true_seq)
                loss.backward()
                optimizer.step()
                epoch_loss += loss.item()
            
            scheduler.step()
            avg_loss = epoch_loss / len(dataloader)

            if (epoch + 1) % 10 == 0:
                print(f"Epoch: {epoch+1}/{epochs} | MAE: {avg_loss:.6f}")

            if best_loss - avg_loss > min_delta:
                best_loss = avg_loss               
                patience_counter = 0               
                best_model_weights = copy.deepcopy(model.state_dict()) 
            else:
                patience_counter += 1            
            
            if patience_counter >= patience:
                print(f"\n[!] EARLY STOPPING: Stopped in {epoch+1} epoch.")
                print(f"No better than {min_delta} since {patience} epochs.")
                break 
            
        model.load_state_dict(best_model_weights)
        print(f"Training done. Best weights with MAE loss: {best_loss:.6f} loaded.\n")
        return model
        
    def create_sequences(inputs, targets, seq_len = 600):
        





